# TP — Traitement Distribué avec Apache Spark

---

Une entreprise e-commerce génère chaque jour des millions de transactions :
- *Quel produit rapporte le plus par région ?*
- *Quel vendeur a le meilleur chiffre d'affaires ce trimestre ?*
- *Quelles catégories sont en baisse de volume ?*

Avec un seul ordinateur, analyser des pétaoctets de données prendrait des jours. **Apache Spark** distribue le calcul sur un cluster de machines et répond en secondes.

**Objectifs de ce TP** :
- Comprendre la relation RDD / DataFrame / Spark SQL
- Écrire des pipelines de traitement distribués
- Savoir quand utiliser chaque API

> Ce TP tourne sur **Google Colab** en mode local — les concepts s'appliquent identiquement sur un vrai cluster.

---
## Partie 1 — Théorie

---

### 1.1 — Architecture Spark

Spark fonctionne en mode **maître / travailleurs** :

```
┌─────────────────────────────────────────────────────┐
│                    Driver                           │
│  SparkSession  ──►  DAG Scheduler  ──►  Task Sched │
└──────────────────────────┬──────────────────────────┘
                           │ distribue les tâches
          ┌────────────────┼────────────────┐
          ▼                ▼                ▼
     Executor 1       Executor 2       Executor 3
     [Partition 1]    [Partition 2]    [Partition 3]
     [Partition 4]    [Partition 5]    [Partition 6]
```

| Composant | Rôle |
|---|---|
| **Driver** | Contient ton code, construit le plan d'exécution (DAG) |
| **SparkSession** | Point d'entrée unique depuis Spark 2.0 |
| **Executor** | Processus JVM sur chaque nœud — exécute les tâches |
| **Partition** | Fragment de données traité par 1 executor à la fois |

> **Règle clé** : 1 partition = 1 tâche = 1 executor au moment T

### 1.2 — RDD : le fondement de Spark

**RDD** (*Resilient Distributed Dataset*) est l'abstraction de base : une collection d'objets **immuable**, **partitionnée** sur le cluster, **recalculable** en cas de panne.

#### Deux types d'opérations

| Type | Caractéristique | Exemples |
|---|---|---|
| **Transformations** | *Lazy* — construisent le DAG sans calculer | `map`, `filter`, `flatMap`, `reduceByKey` |
| **Actions** | *Eager* — déclenchent le calcul réel | `collect`, `count`, `take`, `saveAsTextFile` |

#### Lazy Evaluation

```python
rdd = sc.textFile("data.txt")      # rien ne se passe
rdd2 = rdd.flatMap(lambda l: l.split())  # DAG s'enrichit
rdd3 = rdd2.map(lambda m: (m, 1))        # DAG s'enrichit
rdd3.count()                             # ← ICI Spark calcule tout
```

> Avantage : Spark optimise **l'ensemble du pipeline** avant d'exécuter la moindre ligne.

### 1.3 — DataFrame : RDD avec un schéma

Depuis Spark 2.0, le **DataFrame** est l'API recommandée.

```
RDD[Any]          →  pas de schéma, pas d'optimisation
Dataset[T]        →  typé, Scala/Java uniquement
DataFrame         →  Dataset[Row], colonnes nommées et typées
                     = API Python + optimiseur Catalyst
```

**Ce que Catalyst fait pour toi** :
- Réordonne les filtres pour réduire les données le plus tôt possible
- Fusionne les étapes compatibles
- Choisit le meilleur type de jointure

```python
# RDD : tu décris COMMENT
rdd.filter(lambda r: r[2] > 10).map(lambda r: (r[0], r[2]*r[3]))

# DataFrame : tu décris QUOI — Catalyst décide COMMENT
df.filter("quantite > 10").withColumn("montant", col("quantite") * col("prix"))
```

### 1.4 — Comparaison RDD vs DataFrame

| Critère | RDD | DataFrame |
|---|---|---|
| **Niveau** | Bas niveau | Haut niveau |
| **Schéma** | Aucun | Colonnes nommées et typées |
| **Optimisation** | Manuelle | Automatique (Catalyst) |
| **API** | `map`, `filter`, `reduceByKey`… | `select`, `groupBy`, `join`… |
| **SQL** | Non | Oui (`spark.sql()`) |
| **Performance** | Bonne | Meilleure (en général) |
| **Cas d'usage** | Texte non structuré, logique complexe | Données tabulaires (CSV, Parquet, BDD) |

> **Règle pratique** : utilise **DataFrame par défaut**. Passe en RDD uniquement quand l'API DataFrame ne suffit pas.

### 1.5 — Pipeline complet

```
Données brutes (CSV / JSON / Parquet / BDD)
         ↓  spark.read
     DataFrame
         ↓  filter / withColumn / groupBy / join
  DataFrame transformé
         ↓  createOrReplaceTempView
     Vue SQL temporaire  ──►  spark.sql("SELECT ...")
         ↓  write / show / collect
     Résultat final
```

Les DataFrames et Spark SQL sont **équivalents** — Catalyst les optimise de la même façon. Utilise celui qui est le plus lisible selon le contexte.

---
## Partie 2 — Pratique

---

### Étape 0 — Installation et configuration

In [ ]:
# Installation de PySpark sur Colab
!pip install pyspark --quiet
print("PySpark installé.")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import random, time
from datetime import date, timedelta

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (StructType, StructField,
                                IntegerType, StringType, DoubleType, DateType)
from pyspark.sql.window import Window
from pyspark import SparkContext, StorageLevel
print("Imports OK.")

---
### Étape 1 — SparkSession et SparkContext

**SparkSession** est le point d'entrée unique depuis Spark 2.0.
Il donne accès à l'API DataFrame, SQL, et au SparkContext (API RDD).

In [ ]:
spark = (SparkSession.builder
         .appName("TP-Spark")
         .master("local[*]")          # utilise tous les cœurs disponibles
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())

sc = spark.sparkContext              # accès à l'API RDD
sc.setLogLevel("ERROR")             # réduit le bruit dans les logs

print(f"Spark version : {spark.version}")
print(f"Maître       : {sc.master}")
print(f"App name     : {sc.appName}")

---
### Étape 2 — Génération du dataset

On crée un dataset **ventes e-commerce** (1 000 lignes) avec les colonnes :
`id`, `produit`, `categorie`, `quantite`, `prix_unitaire`, `region`, `vendeur`, `date`

In [ ]:
# Génération du dataset synthétique
random.seed(42)
np.random.seed(42)

produits = {
    "Laptop":      ("Informatique", 850),
    "Smartphone":  ("Informatique", 650),
    "Tablette":    ("Informatique", 420),
    "Casque":      ("Audio",        120),
    "Enceinte":    ("Audio",         80),
    "Montre":      ("Accessoires",  200),
    "Sac":         ("Accessoires",   60),
    "Clavier":     ("Périphériques", 75),
    "Souris":      ("Périphériques", 35),
    "Écran":       ("Informatique", 320),
}
regions  = ["Nord", "Sud", "Est", "Ouest", "Centre"]
vendeurs = ["Alice", "Bob", "Carol", "David", "Eve",
            "Frank", "Grace", "Hector"]

debut = date(2024, 1, 1)

rows = []
for i in range(1, 1001):
    prod = random.choice(list(produits))
    cat, prix_base = produits[prod]
    prix = round(prix_base * random.uniform(0.9, 1.15), 2)
    qte  = random.randint(1, 20)
    j    = random.randint(0, 364)
    d    = debut + timedelta(days=j)
    rows.append({
        "id":          i,
        "produit":     prod,
        "categorie":   cat,
        "quantite":    qte,
        "prix_unitaire": prix,
        "region":      random.choice(regions),
        "vendeur":     random.choice(vendeurs),
        "date":        d.isoformat(),
        "mois":        d.month,
        "trimestre":   f"T{(d.month-1)//3 + 1}",
    })

df_pd = pd.DataFrame(rows)
df_pd.to_csv("ventes.csv", index=False)
print(f"Dataset créé : {len(df_pd)} lignes × {len(df_pd.columns)} colonnes")
df_pd.head(3)

---
### Étape 3 — API RDD

On commence par le bas niveau pour bien comprendre ce qui se passe sous le capot.

#### 3.1 — Créer un RDD

In [ ]:
# Méthode 1 : depuis une liste Python
rdd_nums = sc.parallelize([10, 25, 3, 47, 8, 99, 14, 66])
print("Type        :", type(rdd_nums))
print("Partitions  :", rdd_nums.getNumPartitions())
print("Contenu     :", rdd_nums.collect())

In [ ]:
# Méthode 2 : depuis un fichier
rdd_csv = sc.textFile("ventes.csv")
print("Nombre de lignes :", rdd_csv.count())
print("Premières lignes :")
for l in rdd_csv.take(3):
    print(" ", l)

#### 3.2 — Transformations : `map`, `filter`, `flatMap`

In [ ]:
# map : applique une fonction à chaque élément
rdd_double = rdd_nums.map(lambda x: x * 2)
print("map(x*2)   :", rdd_double.collect())

# filter : garde les éléments qui satisfont la condition
rdd_pairs = rdd_nums.filter(lambda x: x % 2 == 0)
print("filter(pair):", rdd_pairs.collect())

In [ ]:
# flatMap : map + aplatissement
phrases = sc.parallelize(["Spark est rapide", "Spark est distribué", "Python est simple"])

mots = phrases.flatMap(lambda p: p.split())
print("flatMap (mots) :", mots.collect())

# Compter les mots uniques
print("Mots uniques   :", mots.distinct().collect())

#### 3.3 — Agrégations : `reduceByKey` vs `groupByKey`

In [ ]:
# reduceByKey : réduit localement AVANT le shuffle → plus efficace
paires = sc.parallelize([
    ("Nord", 150), ("Sud", 200), ("Nord", 320),
    ("Est",   90), ("Sud", 110), ("Est",  250),
])

# groupByKey : regroupe toutes les valeurs par clé (shuffle coûteux)
# reduceByKey : réduit localement d'abord, puis shuffle (recommandé)
total_reduce = paires.reduceByKey(lambda a, b: a + b)
print("reduceByKey (CA par région) :", total_reduce.collect())

In [ ]:
# Chaîner plusieurs transformations
top_regions = (paires
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: x[1], ascending=False)
)
print("Régions triées par CA :")
for region, ca in top_regions.collect():
    print(f"  {region:<10} : {ca}")

#### 3.4 — WordCount complet

Le "Hello World" de Spark : compter les occurrences de chaque produit.

In [ ]:
# Charger le CSV, sauter l'en-tête, extraire la colonne produit (index 1)
header = rdd_csv.first()

wordcount = (rdd_csv
    .filter(lambda l: l != header)                  # enlever l'en-tête
    .map(lambda l: l.split(",")[1])                 # colonne "produit"
    .map(lambda prod: (prod, 1))                    # paires (produit, 1)
    .reduceByKey(lambda a, b: a + b)                # comptage
    .sortBy(lambda x: x[1], ascending=False)        # tri décroissant
)

print("Nombre de ventes par produit :")
for produit, count in wordcount.collect():
    print(f"  {produit:<15} : {count} ventes")

#### 3.5 — Lazy Evaluation : observation directe

In [ ]:
# Les transformations ne calculent rien — elles construisent le DAG
rdd_lazy = (sc.parallelize(range(1_000_000))
              .filter(lambda x: x % 2 == 0)
              .map(lambda x: x ** 2))

print("Après les transformations : rien n'a encore été calculé.")
print("Type :", type(rdd_lazy))

# L'action déclenche le calcul
debut = time.time()
total = rdd_lazy.sum()
fin   = time.time()
print(f"\nAprès sum() : calcul effectué en {fin-debut:.3f}s")
print(f"Somme des carrés pairs (0 à 999 999) : {total:,}")

---
### Étape 4 — API DataFrame

Même objectif que l'étape 3, mais avec l'API haut niveau. Catalyst optimise automatiquement.

#### 4.1 — Lecture et exploration

In [ ]:
# Lecture du CSV avec inférence de schéma
df = spark.read.option("header", True).option("inferSchema", True).csv("ventes.csv")

print(f"Dimensions : {df.count()} lignes × {len(df.columns)} colonnes")
print()
df.printSchema()

In [ ]:
# Aperçu des données
df.show(5, truncate=False)

In [ ]:
# Statistiques descriptives
df.select("quantite", "prix_unitaire").describe().show()

#### 4.2 — Sélection et filtrage

In [ ]:
# select : choisir les colonnes
df.select("produit", "region", "quantite", "prix_unitaire").show(5)

In [ ]:
# filter : garder les lignes qui satisfont une condition
df_informatique = df.filter(F.col("categorie") == "Informatique")
print(f"Ventes Informatique : {df_informatique.count()} lignes")
df_informatique.show(4)

In [ ]:
# Conditions combinées
df_filtre = df.filter(
    (F.col("quantite") > 10) & (F.col("prix_unitaire") > 300)
)
print(f"Lignes (qte>10 ET prix>300) : {df_filtre.count()}")
df_filtre.select("produit", "quantite", "prix_unitaire", "region").show(5)

#### 4.3 — Nouvelles colonnes avec `withColumn`

In [ ]:
# Calculer le montant de chaque vente
df = df.withColumn("montant", F.round(F.col("quantite") * F.col("prix_unitaire"), 2))

# Catégorie de quantité
df = df.withColumn("volume",
    F.when(F.col("quantite") <= 5,  "Faible")
     .when(F.col("quantite") <= 12, "Moyen")
     .otherwise("Élevé")
)

df.select("produit", "quantite", "prix_unitaire", "montant", "volume").show(5)

#### 4.4 — Agrégations avec `groupBy`

In [ ]:
# CA total par catégorie
df.groupBy("categorie").agg(
    F.round(F.sum("montant"), 2).alias("ca_total"),
    F.round(F.avg("prix_unitaire"), 2).alias("prix_moyen"),
    F.count("*").alias("nb_ventes")
).orderBy(F.col("ca_total").desc()).show()

In [ ]:
# Meilleurs vendeurs
df.groupBy("vendeur").agg(
    F.round(F.sum("montant"), 2).alias("ca_total"),
    F.countDistinct("produit").alias("produits_differents"),
    F.count("*").alias("nb_transactions")
).orderBy(F.col("ca_total").desc()).show()

In [ ]:
# Pivot : CA par région et trimestre
df.groupBy("region")   .pivot("trimestre", ["T1", "T2", "T3", "T4"])   .agg(F.round(F.sum("montant"), 0))   .orderBy("region")   .show()

#### 4.5 — Jointures

In [ ]:
# Créer une table de référence des catégories avec objectif CA
objectifs = spark.createDataFrame([
    ("Informatique",  150_000.0),
    ("Audio",          40_000.0),
    ("Accessoires",    30_000.0),
    ("Périphériques",  20_000.0),
], ["categorie", "objectif_ca"])

# Joindre avec les ventes agrégées
ca_par_cat = df.groupBy("categorie").agg(
    F.round(F.sum("montant"), 2).alias("ca_reel")
)

resultat = ca_par_cat.join(objectifs, on="categorie", how="left")
resultat = resultat.withColumn("atteint_%",
    F.round(F.col("ca_reel") / F.col("objectif_ca") * 100, 1))
resultat.orderBy(F.col("atteint_%").desc()).show()

---
### Étape 5 — Spark SQL

Spark SQL permet d'interroger un DataFrame avec du **SQL standard**.
Le résultat est toujours un DataFrame — les deux APIs sont interchangeables.

In [ ]:
# Enregistrer les DataFrames comme vues SQL temporaires
df.createOrReplaceTempView("ventes")
objectifs.createOrReplaceTempView("objectifs")
print("Vues créées : ventes, objectifs")

In [ ]:
# Requête simple
spark.sql('''
    SELECT produit, COUNT(*) AS nb_ventes, ROUND(SUM(montant), 2) AS ca_total
    FROM ventes
    GROUP BY produit
    ORDER BY ca_total DESC
    LIMIT 5
''').show()

In [ ]:
# Requête avec JOIN et filtre
spark.sql('''
    SELECT
        v.region,
        v.categorie,
        ROUND(SUM(v.montant), 2)                  AS ca_reel,
        o.objectif_ca,
        ROUND(SUM(v.montant) / o.objectif_ca * 100, 1) AS pct_objectif
    FROM ventes v
    JOIN objectifs o ON v.categorie = o.categorie
    WHERE v.trimestre IN ('T1', 'T2')
    GROUP BY v.region, v.categorie, o.objectif_ca
    ORDER BY pct_objectif DESC
    LIMIT 10
''').show()

In [ ]:
# Sous-requête : vendeurs au-dessus de la moyenne
spark.sql('''
    SELECT vendeur, ROUND(ca_total, 2) AS ca_total
    FROM (
        SELECT vendeur, SUM(montant) AS ca_total
        FROM ventes
        GROUP BY vendeur
    )
    WHERE ca_total > (SELECT AVG(ca_total) FROM (
        SELECT vendeur, SUM(montant) AS ca_total
        FROM ventes GROUP BY vendeur
    ))
    ORDER BY ca_total DESC
''').show()

---
### Étape 6 — Passage RDD ↔ DataFrame

Les deux APIs sont interopérables.

In [ ]:
# DataFrame → RDD
rdd_from_df = df.select("produit", "montant").rdd
print("Type :", type(rdd_from_df))
print("Premiers éléments :")
for row in rdd_from_df.take(3):
    print(" ", row)           # chaque élément est une Row

In [ ]:
# Traitement RDD sur données DataFrame
top3_produits = (rdd_from_df
    .map(lambda r: (r["produit"], r["montant"]))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: x[1], ascending=False)
    .take(3)
)
print("Top 3 produits (via RDD) :")
for prod, ca in top3_produits:
    print(f"  {prod:<15} : {ca:,.2f} €")

In [ ]:
# RDD → DataFrame (avec schéma explicite)
rdd_data = sc.parallelize([
    (1, "Paris",  12500.0),
    (2, "Lyon",    8300.0),
    (3, "Bordeaux", 5100.0),
])

schema = StructType([
    StructField("id",     IntegerType(), False),
    StructField("ville",  StringType(),  True),
    StructField("budget", DoubleType(),  True),
])

df_from_rdd = spark.createDataFrame(rdd_data, schema)
df_from_rdd.show()
df_from_rdd.printSchema()

---
### Étape 7 — Cache et Persistance

Quand un DataFrame est **réutilisé plusieurs fois**, le mettre en cache évite de recalculer le pipeline depuis le début.

#### 7.1 — Sans cache

In [ ]:
# Pipeline un peu coûteux
df_calcul = (df
    .filter(F.col("categorie") == "Informatique")
    .withColumn("montant_ttc", F.col("montant") * 1.2)
    .groupBy("produit", "trimestre")
    .agg(F.round(F.sum("montant_ttc"), 2).alias("ca_ttc"))
)

# Exécution 1 — calcule depuis le début
t0 = time.time()
n1 = df_calcul.count()
t1 = time.time()
print(f"Sans cache — 1er appel  : {t1-t0:.3f}s  ({n1} lignes)")

# Exécution 2 — recalcule depuis le début
t0 = time.time()
n2 = df_calcul.count()
t1 = time.time()
print(f"Sans cache — 2ème appel : {t1-t0:.3f}s  ({n2} lignes)")

#### 7.2 — Avec cache

In [ ]:
# Même pipeline — avec .cache()
df_cache = (df
    .filter(F.col("categorie") == "Informatique")
    .withColumn("montant_ttc", F.col("montant") * 1.2)
    .groupBy("produit", "trimestre")
    .agg(F.round(F.sum("montant_ttc"), 2).alias("ca_ttc"))
    .cache()
)

# 1er appel : calcule ET met en cache
t0 = time.time()
n1 = df_cache.count()
t1 = time.time()
print(f"Avec cache — 1er appel  : {t1-t0:.3f}s  (calcul + mise en cache)")

# 2ème appel : lit depuis le cache
t0 = time.time()
n2 = df_cache.count()
t1 = time.time()
print(f"Avec cache — 2ème appel : {t1-t0:.3f}s  (lecture cache) ✓")

df_cache.unpersist()   # libérer la mémoire
print("Cache libéré.")

#### 7.3 — Niveaux de persistance

In [ ]:
# Persistance avec niveau explicite
from pyspark import StorageLevel

df_persist = df.filter(F.col("region") == "Nord").persist(
    StorageLevel.MEMORY_AND_DISK   # RAM d'abord, disque si insuffisant
)

df_persist.count()   # déclenche le calcul + persistance

print("Niveaux disponibles :")
for level in ["MEMORY_ONLY", "MEMORY_AND_DISK", "DISK_ONLY", "MEMORY_ONLY_2"]:
    print(f"  StorageLevel.{level}")

df_persist.unpersist()

---
### Bilan

| Concept | Rôle | Quand l'utiliser |
|---|---|---|
| **RDD** | Collection distribuée bas niveau | Texte brut, logique complexe, interop legacy |
| **DataFrame** | Table distribuée avec schéma | Données tabulaires — usage principal |
| **Spark SQL** | SQL sur DataFrame | Requêtes lisibles, agrégations complexes |
| **Cache** | Stocke un résultat intermédiaire | DataFrame réutilisé plusieurs fois |
| **Lazy eval** | Calcul différé jusqu'à l'action | Optimisation automatique du pipeline |

#### Règles à retenir

1. **DataFrame par défaut** — passe en RDD seulement si nécessaire
2. **`reduceByKey` plutôt que `groupByKey`** — réduit avant le shuffle
3. **`cache()` quand tu réutilises** — évite de recalculer le pipeline
4. **`inferSchema=False` en production** — définis le schéma explicitement
5. **Spark SQL et l'API DataFrame sont équivalents** — utilise celui qui est le plus lisible

```python
spark.stop()   # toujours fermer la session en fin de TP
```

In [ ]:
spark.stop()
print("Session Spark fermée.")